In [1]:
#%pip install optuna
#%pip install darts
#%pip install pytorch-lightning

In [2]:
# ============================================================
# PAPER-STYLE LSTM TUNING SCRIPT
# Saves best parameters to ../Outputs/paper_lstm.pkl
# ============================================================

import os
import sys
import json
import pickle
import warnings
import datetime
from pathlib import Path
import pytorch_lightning as pl

In [3]:
# ============================================================
# PAPER-STYLE LSTM PARAMETER SEARCH (MULTI-REGION)
# - matches paper logic more closely:
#   * regularize to daily frequency with asfreq("D")
#   * interpolate missing dates
#   * build Darts TimeSeries with freq="D"
# - adds robustness:
#   * checks for NaN / inf
#   * converts invalid trial scores to +inf
#   * skips impossible folds
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
import os
import math
import pickle
import random
import warnings
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import torch

from sklearn.preprocessing import StandardScaler

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.metrics import rmse
from darts.models import BlockRNNModel

from pytorch_lightning.callbacks import EarlyStopping
warnings.filterwarnings("ignore")

# optional pruning
try:
    from optuna.integration import PyTorchLightningPruningCallback
    HAS_PRUNING = True
except Exception:
    HAS_PRUNING = False

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


In [ ]:
# ============================================================
# PAPER-STYLE LSTM PARAMETER SEARCH (MULTI-REGION, PER-REGION PKL)
# - regularizes each region to daily frequency with interpolation
# - performs nested CV + Optuna search per region
# - saves:
#     1) one combined trials CSV across all regions/folds/trials
#     2) one PKL file per region with best parameters
# ============================================================

# ============================================================
# IMPORTS
# ============================================================
import os
import pickle
import random
import warnings
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import optuna
import torch

from sklearn.preprocessing import StandardScaler

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.metrics import rmse
from darts.models import BlockRNNModel

from pytorch_lightning.callbacks import EarlyStopping

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

# optional pruning
try:
    from optuna.integration import PyTorchLightningPruningCallback
    HAS_PRUNING = True
except Exception:
    HAS_PRUNING = False


# ============================================================
# CONFIG
# ============================================================
INPUT_CSV = "../EDA/region_temp_extended.csv"
OUTPUT_DIR = Path("../Outputs/lstm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_TRIALS_CSV = OUTPUT_DIR / "paper_lstm_trials.csv"

TODAY = datetime.datetime.today().strftime("%Y-%m-%d")
START = datetime.datetime.now()

DATE_COL = "date"
REGION_CANDIDATES = ["region_code", "name"]
TARGET_COL = "de_trend_seas"
COVARIATE_COLS = ["dayofyear", "month", "dts_doyavge", "dts_doyvar"]

# paper-style CV settings
N_FOLDS = 3
VAL_SIZE_DAYS = 365 * 5

# optuna settings
N_TRIALS = 20
RANDOM_STATE = 42

# final model settings metadata
FINAL_N_EPOCHS = 100
BATCH_SIZE = 1024

# cpu settings
N_THREADS = 8
os.environ["OMP_NUM_THREADS"] = str(N_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_THREADS)
os.environ["MKL_NUM_THREADS"] = str(N_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(N_THREADS)
os.environ["TORCH_NUM_THREADS"] = str(N_THREADS)

torch.set_num_threads(N_THREADS)

# reproducibility
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass


# ============================================================
# HELPERS
# ============================================================
def find_region_col(df: pd.DataFrame) -> str:
    for c in REGION_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Could not find region column. Expected one of {REGION_CANDIDATES}")


def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])

    region_col = find_region_col(df)

    required = {DATE_COL, region_col, TARGET_COL, *COVARIATE_COLS}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    keep_cols = [DATE_COL, region_col, TARGET_COL, *COVARIATE_COLS]
    df = df[keep_cols].copy()

    numeric_cols = [TARGET_COL] + COVARIATE_COLS
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df = df.sort_values([region_col, DATE_COL]).reset_index(drop=True)
    return df


def ts_is_finite(ts: TimeSeries) -> bool:
    vals = ts.values(copy=False)
    return np.isfinite(vals).all()


def safe_float(x) -> float:
    try:
        x = float(x)
        return x if np.isfinite(x) else float("inf")
    except Exception:
        return float("inf")


def safe_rmse(actual: TimeSeries, pred: TimeSeries) -> float:
    try:
        return safe_float(rmse(actual, pred))
    except Exception:
        return float("inf")


def time_series_cv(
    series: TimeSeries,
    covariates: TimeSeries,
    n_folds: int = 3,
    val_size: int = 365 * 5,
):
    folds = []
    total_len = len(series)

    for i in range(n_folds):
        split_point = total_len - (n_folds - i) * val_size

        train = series[:split_point]
        val = series[split_point:split_point + val_size]

        cov_train = covariates[:split_point]
        cov_val = covariates[split_point:split_point + val_size]

        if len(train) > 0 and len(val) > 0:
            folds.append((train, val, cov_train, cov_val))

    return folds


def build_series_for_region(region_df: pd.DataFrame):
    """
    Paper-style preprocessing:
      1. sort by date
      2. set daily frequency with asfreq("D")
      3. interpolate missing dates/values
      4. create target + covariate series
      5. scale target and covariates separately
    """
    region_df = region_df.sort_values(DATE_COL).copy()

    needed_cols = [DATE_COL, TARGET_COL] + COVARIATE_COLS
    region_df = region_df[needed_cols].copy()

    # regularize to daily calendar, then interpolate like paper code
    region_df = region_df.set_index(DATE_COL).asfreq("D")

    region_df[[TARGET_COL] + COVARIATE_COLS] = (
        region_df[[TARGET_COL] + COVARIATE_COLS]
        .interpolate(method="linear", limit_direction="both", axis=0)
    )

    region_df = region_df.reset_index()

    # final cleanup
    before = len(region_df)
    region_df = region_df.replace([np.inf, -np.inf], np.nan)
    region_df = region_df.dropna(subset=[TARGET_COL] + COVARIATE_COLS).copy()
    dropped = before - len(region_df)

    if dropped > 0:
        print(f"  Dropped {dropped} rows after interpolation/cleaning")

    if len(region_df) == 0:
        raise ValueError("No valid rows remain after interpolation and cleaning.")

    if region_df[TARGET_COL].nunique() <= 1:
        raise ValueError("Target column is constant after cleaning.")

    target_series = TimeSeries.from_dataframe(
        df=region_df,
        time_col=DATE_COL,
        value_cols=TARGET_COL,
        fill_missing_dates=False,
        freq="D",
    )

    cov_series = TimeSeries.from_dataframe(
        df=region_df,
        time_col=DATE_COL,
        value_cols=COVARIATE_COLS,
        fill_missing_dates=False,
        freq="D",
    )

    target_scaler = Scaler(scaler=StandardScaler())
    target_scaled = target_scaler.fit_transform(target_series)

    cov_scaler = Scaler(scaler=StandardScaler())
    cov_scaled = cov_scaler.fit_transform(cov_series)

    if not ts_is_finite(target_scaled):
        raise ValueError("Scaled target series contains NaN/inf values.")
    if not ts_is_finite(cov_scaled):
        raise ValueError("Scaled covariate series contains NaN/inf values.")

    return target_series, cov_series, target_scaled, cov_scaled, target_scaler, cov_scaler


def make_objective(train, val, cov_train, cov_val, cov_scaled_full):
    def objective(trial):
        try:
            callbacks = []

            early_stop_callback = EarlyStopping(
                monitor="val_loss",
                patience=3,
                min_delta=0.001,
                mode="min",
                verbose=False,
            )
            callbacks.append(early_stop_callback)

            if HAS_PRUNING:
                pruning_callback = PyTorchLightningPruningCallback(
                    trial, monitor="val_loss"
                )
                callbacks.append(pruning_callback)

            days_in = trial.suggest_int("days_in", 2, 3)
            in_len = 365 * days_in
            out_len = 1

            if len(train) <= in_len:
                return float("inf")

            lr = trial.suggest_float("lr", 5e-5, 1e-3, log=True)
            hidden_dim = trial.suggest_categorical("hidden_dim", [4, 8, 16, 32, 128])
            n_rnn_layers = trial.suggest_int("n_rnn_layers", 1, 2)
            dropout = trial.suggest_float("dropout", 0.0, 0.4)
            activation = trial.suggest_categorical("activation", ["ReLU", "Tanh"])

            trial.set_user_attr("input_chunk_length", in_len)
            trial.set_user_attr("output_chunk_length", out_len)

            model = BlockRNNModel(
                input_chunk_length=in_len,
                output_chunk_length=out_len,
                batch_size=BATCH_SIZE,
                n_epochs=10,
                nr_epochs_val_period=1,
                random_state=RANDOM_STATE,
                model="LSTM",
                hidden_dim=hidden_dim,
                n_rnn_layers=n_rnn_layers,
                dropout=dropout,
                activation=activation,
                optimizer_kwargs={"lr": lr},
                likelihood=None,
                force_reset=True,
                save_checkpoints=False,
                pl_trainer_kwargs={
                    "callbacks": callbacks,
                    "enable_progress_bar": False,
                    "accelerator": "cpu",
                    "devices": 1,
                    "default_root_dir": str(OUTPUT_DIR),
                    "gradient_clip_val": 1.0,
                    "log_every_n_steps": 999999,
                    "enable_model_summary": False,
                    "logger": False,
                },
                model_name=f"paper_lstm_trial_{trial.number}",
            )

            model.fit(
                series=train,
                past_covariates=cov_train,
                val_series=val,
                val_past_covariates=cov_val,
                verbose=False,
                dataloader_kwargs={"num_workers": 0},
            )

            forecast = model.predict(
                n=len(val),
                series=train,
                past_covariates=cov_scaled_full,
                verbose=False,
                show_warnings=False,
            )

            if forecast is None:
                return float("inf")
            if not ts_is_finite(forecast):
                return float("inf")
            if not ts_is_finite(val):
                return float("inf")

            return safe_rmse(val, forecast)

        except Exception as e:
            print(f"  Trial {trial.number} failed: {e}")
            return float("inf")

    return objective


# ============================================================
# MAIN
# ============================================================
def main():
    df = load_data(INPUT_CSV)
    region_col = find_region_col(df)

    all_trial_rows = []

    regions = list(df[region_col].dropna().unique())
    print(f"Found {len(regions)} regions")

    for region in regions:
        print(f"\n=== REGION {region} ===")

        region_df = df[df[region_col] == region].copy().sort_values(DATE_COL)

        raw_nan = region_df[TARGET_COL].isna().sum()
        raw_inf = np.isinf(region_df[TARGET_COL].to_numpy(dtype=float)).sum()
        print(f"  Raw rows: {len(region_df)} | target NaN: {raw_nan} | target inf: {raw_inf}")

        min_required = N_FOLDS * VAL_SIZE_DAYS + 365 * 3 + 1
        if len(region_df) < min_required:
            print(f"Skipping region {region}: not enough raw data ({len(region_df)} rows)")
            continue

        try:
            (
                target_series,
                cov_series,
                target_scaled,
                cov_scaled,
                target_scaler,
                cov_scaler,
            ) = build_series_for_region(region_df)
        except Exception as e:
            print(f"Skipping region {region}: data issue -> {e}")
            continue

        print(f"  Clean/regularized rows: {len(target_scaled)}")

        folds = time_series_cv(
            target_scaled,
            cov_scaled,
            n_folds=N_FOLDS,
            val_size=VAL_SIZE_DAYS,
        )

        if len(folds) == 0:
            print(f"Skipping region {region}: no valid folds")
            continue

        best_models_info = []

        for fold_idx, (train, val, cov_train, cov_val) in enumerate(folds, start=1):
            print(
                f"  Region {region} | Fold {fold_idx}/{len(folds)} "
                f"| train={len(train)} | val={len(val)}"
            )

            if len(train) <= 365 * 2:
                print(f"  Skipping fold {fold_idx}: train too short")
                continue
            if not ts_is_finite(train):
                print(f"  Skipping fold {fold_idx}: train contains NaN/inf")
                continue
            if not ts_is_finite(val):
                print(f"  Skipping fold {fold_idx}: val contains NaN/inf")
                continue
            if not ts_is_finite(cov_train):
                print(f"  Skipping fold {fold_idx}: cov_train contains NaN/inf")
                continue
            if not ts_is_finite(cov_val):
                print(f"  Skipping fold {fold_idx}: cov_val contains NaN/inf")
                continue

            study = optuna.create_study(
                direction="minimize",
                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
            )

            study.optimize(
                make_objective(train, val, cov_train, cov_val, cov_scaled),
                n_trials=N_TRIALS,
                show_progress_bar=False,
            )

            completed_trials = [
                t for t in study.trials
                if t.value is not None and np.isfinite(t.value)
            ]

            if len(completed_trials) == 0:
                print(f"  No successful trials for region {region}, fold {fold_idx}")
                continue

            best_trial = min(completed_trials, key=lambda t: t.value)

            best_models_info.append(
                {
                    "fold": fold_idx,
                    "rmse": float(best_trial.value),
                    "params": best_trial.params.copy(),
                    "in_len": best_trial.user_attrs["input_chunk_length"],
                    "out_len": best_trial.user_attrs["output_chunk_length"],
                }
            )

            print(
                f"    Best fold trial RMSE: {float(best_trial.value):.6f} "
                f"| params: {best_trial.params}"
            )

            for trial in study.trials:
                row = {
                    "region": region,
                    "fold": fold_idx,
                    "trial": trial.number,
                    "status": trial.state.name,
                    "rmse": (
                        float(trial.value)
                        if trial.value is not None and np.isfinite(trial.value)
                        else np.nan
                    ),
                }
                row.update(trial.params)
                all_trial_rows.append(row)

        if len(best_models_info) == 0:
            print(f"No successful folds for region {region}")
            continue

        best_model_info = min(best_models_info, key=lambda x: x["rmse"])

        final_best_params = best_model_info["params"].copy()
        final_lr = final_best_params.get("lr", None)

        region_payload = {
            "created_at": datetime.datetime.now().isoformat(),
            "input_csv": INPUT_CSV,
            "region": region,
            "best_fold_rmse": best_model_info["rmse"],
            "best_fold": best_model_info["fold"],
            "best_params": final_best_params,
            "input_chunk_length": best_model_info["in_len"],
            "output_chunk_length": best_model_info["out_len"],
            "final_n_epochs": FINAL_N_EPOCHS,
            "batch_size": BATCH_SIZE,
            "covariate_cols": COVARIATE_COLS,
            "target_col": TARGET_COL,
            "learning_rate": final_lr,
            "n_folds": N_FOLDS,
            "val_size_days": VAL_SIZE_DAYS,
            "n_trials": N_TRIALS,
        }

        print("  Best region params:")
        print(f"  {region_payload}")

        region_pkl = OUTPUT_DIR / f"paper_lstm_params_{region}.pkl"
        with open(region_pkl, "wb") as f:
            pickle.dump(region_payload, f)

        print(f"  Saved region parameters to {region_pkl}")

    if len(all_trial_rows) > 0:
        trials_df = pd.DataFrame(all_trial_rows)
        trials_df.to_csv(OUTPUT_TRIALS_CSV, index=False)
        print(f"\nSaved trials to {OUTPUT_TRIALS_CSV}")

    print(f"Total runtime: {datetime.datetime.now() - START}")


if __name__ == "__main__":
    main()

[I 2026-03-27 17:04:47,489] A new study created in memory with name: no-name-520eac33-d6a9-48ea-b847-f4868b5844ed


Found 8 regions

=== REGION 11 ===
  Raw rows: 18250 | target NaN: 0 | target inf: 0
  Clean/regularized rows: 18263
  Region 11 | Fold 1/3 | train=12788 | val=1825


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer.fit` stopped: `max_epochs=10` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
[I 2026-03-27 17:07:22,279] Trial 0 finished with value: 0.9934168172492317 and parameters: {'days_in': 2, 'lr': 0.0008627358286640176, 'hidden_dim': 4, 'n_rnn_layers': 2, 'dropout': 0.24044600469728353, 'activation': 'ReLU'}. Best is trial 0 with value: 0.9934168172492317.
GPU available: True (mps), used: False
TPU available: Fal